# 01 — Top-down pipeline only

Questo notebook esegue **solo la prima parte/top-down** della pipeline GORE. La generazione di description, actors, HLG e LLG usa i generatori originali; non viene eseguito il ciclo bottom-up.

Configurazione sperimentale usata qui: **Actors Few-shot senza evaluator**; **HLG e LLG Zero-shot con evaluator/reflection loop**. La soglia e il numero massimo di iterazioni sono letti direttamente da `refine_response.py` (8.5 e 3 nella configurazione originale).

Gli output strutturati vengono salvati in `output/top_down/` e possono essere riutilizzati dal notebook bottom-up e dal notebook di valutazione.


In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from key import get_key_openai, get_key_llama, count_Llama_keys

openai_key = get_key_openai()
groq_key = get_key_llama()

print("Chiave OpenAI presente:", bool(openai_key))
print("Formato OpenAI plausibile:", openai_key.startswith("sk-") if openai_key else False)
print("Chiave Groq presente:", bool(groq_key))
print("Formato Groq plausibile:", groq_key.startswith("gsk_") if groq_key else False)
print("Numero chiavi Groq:", count_Llama_keys())

Project root: C:\Users\agnes_lryeu3v\Desktop\tesi
Chiave OpenAI presente: True
Formato OpenAI plausibile: True
Chiave Groq presente: True
Formato Groq plausibile: True
Numero chiavi Groq: 5


## 1. Configurazione

La documentazione segue il percorso originale: se la ground truth contiene `link-readme`, viene chiamato `generate_description(...)`; altrimenti viene usata direttamente la descrizione disponibile nella ground truth. Non viene introdotto chunking aggiuntivo nella baseline.


In [4]:
import json

from groundtruth import (
    GENOME,
    GESTAO_HOSPITAL,
    SIA_PROJECT_25_26,
    LONDON_AMBULANCE_SYSTEM,
)
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
)
from src.self_critique.refine_response import (
    EvalMode,
    MAX_ATTEMPTS,
    QUALITY_THRESHOLD,
    generate_response_with_reflection,
)
from src.examples.shot_learning import ShotPromptingMode
from src.utils import get_markdown
import src.llm_clients as llm_clients

GROUNDTRUTHS = [
    SIA_PROJECT_25_26,
    GENOME,
    GESTAO_HOSPITAL,
    LONDON_AMBULANCE_SYSTEM,
]

ACTOR_PROMPTING_MODE = ShotPromptingMode.FEW_SHOT
GOAL_PROMPTING_MODE = ShotPromptingMode.ZERO_SHOT
GOAL_EVALUATOR_ABLATION = False
BASE_CONFIG_NAME = "ZERO_SHOT_feedback_original_top_down"

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_DIR.mkdir(parents=True, exist_ok=True)

def top_down_output_file_for(dataset_name: str) -> Path:
    return TOP_DOWN_DIR / f"{dataset_name}_{BASE_CONFIG_NAME}_top_down.json"

print("Top-down output directory:", TOP_DOWN_DIR)
print("OpenAI structured generator:", llm_clients.OPENAI_STRUCTURED_MODEL)
print("OpenAI text generator:", llm_clients.OPENAI_TEXT_MODEL)
print("Preferred Groq evaluator:", llm_clients.PREFERRED_GROQ_EVALUATOR_MODEL)
print("Groq fallback evaluator:", llm_clients.GROQ_EVALUATOR_FALLBACK_MODEL)
print("Quality threshold:", QUALITY_THRESHOLD)
print("Max reflection attempts:", MAX_ATTEMPTS)


Top-down output directory: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down
OpenAI structured generator: gpt-4o-mini
OpenAI text generator: gpt-4o
Preferred Groq evaluator: llama-3.3-70b-versatile
Groq fallback evaluator: qwen/qwen3.6-27b
Quality threshold: 8.5
Max reflection attempts: 3


## 2. Esecuzione top-down e salvataggio

Per ogni dataset viene costruita la `description` con il percorso originale. Gli Actors vengono generati in Few-shot senza critic. HLG e LLG vengono generati in Zero-shot e passano attraverso l'evaluator Few-shot di `refine_response.py`. Il JSON conserva sia le liste testuali sia gli oggetti Pydantic strutturati necessari al confronto sperimentale e al successivo bottom-up.


In [5]:
generated_descriptions = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}


def run_top_down_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]

    try:
        # ---------------------------------------------------------------
        # 1. ORIGINAL documentation preprocessing
        # ---------------------------------------------------------------
        if "link-readme" in groundtruth:
            documentation_link = groundtruth["link-readme"]

            # Saved only for traceability; generate_description() below follows
            # the original repository path and reads the README itself.
            raw_description = get_markdown(documentation_link)
            generated_description = generate_description(documentation_link)
            description = generated_description.description

            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "original_generate_description",
            }
        else:
            raw_description = str(groundtruth["description"])
            description = raw_description

            preprocessing_info = {
                "chunked": False,
                "chunkCount": 1,
                "originalChars": len(raw_description),
                "processedChars": len(description),
                "strategy": "groundtruth_description",
            }

        print(
            f"[{dataset_name}] Documentation ready | "
            f"raw={len(raw_description)} chars | "
            f"pipeline={len(description)} chars"
        )

        # ---------------------------------------------------------------
        # 2. ACTORS: Few-shot, no evaluator
        # ---------------------------------------------------------------
        print("Actors STARTING... (Few-shot, no evaluator)")
        actors = generate_actors(
            description,
            feedback=None,
            mode=ACTOR_PROMPTING_MODE,
        )
        print("Actors DONE...")
        print(actors)

        actors_score = None
        actors_critique = None

        # ---------------------------------------------------------------
        # 3. HLG: Zero-shot + evaluator/reflection
        # ---------------------------------------------------------------
        high_level_goals, hl_score, hl_critique = (
            generate_response_with_reflection(
                target_type="High Level Goals",
                call_function=generate_high_level_goals,
                define_args=(description, actors),
                eval_mode=EvalMode.HIGH_LEVEL,
                eval_args=(description, actors),
                shotPromptingMode=GOAL_PROMPTING_MODE,
                llama_ablation=GOAL_EVALUATOR_ABLATION,
            )
        )

        # ---------------------------------------------------------------
        # 4. LLG: Zero-shot + evaluator/reflection
        # ---------------------------------------------------------------
        low_level_goals, ll_score, ll_critique = (
            generate_response_with_reflection(
                target_type="Low Level Goals",
                call_function=generate_low_level_goals,
                define_args=(high_level_goals,),
                eval_mode=EvalMode.LOW_LEVEL,
                eval_args=(description, actors, high_level_goals),
                shotPromptingMode=GOAL_PROMPTING_MODE,
                llama_ablation=GOAL_EVALUATOR_ABLATION,
            )
        )

        generated_descriptions[dataset_name] = description
        generated_actor_objects[dataset_name] = actors
        generated_hl_objects[dataset_name] = high_level_goals
        generated_ll_objects[dataset_name] = low_level_goals

        output = {
            "name": dataset_name,
            "description": description,
            "rawDescription": raw_description,
            "documentationPreprocessing": preprocessing_info,
            "actors": [actor.name for actor in actors.actors],
            "highLevelGoals": [
                goal.description for goal in high_level_goals.goals
            ],
            "lowLevelGoals": [
                goal.description for goal in low_level_goals.low_level_goals
            ],
            "structuredActors": actors.model_dump(mode="json"),
            "structuredHighLevelGoals": high_level_goals.model_dump(mode="json"),
            "structuredLowLevelGoals": low_level_goals.model_dump(mode="json"),
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
            "baselineConfiguration": {
                "generatorStructuredModel": llm_clients.OPENAI_STRUCTURED_MODEL,
                "generatorTextModel": llm_clients.OPENAI_TEXT_MODEL,
                "actorPromptingMode": ACTOR_PROMPTING_MODE.name,
                "actorEvaluatorEnabled": False,
                "goalPromptingMode": GOAL_PROMPTING_MODE.name,
                "highLevelGoalEvaluatorEnabled": True,
                "lowLevelGoalEvaluatorEnabled": True,
                "goalEvaluatorModel": llm_clients.GROQ_EVALUATOR_MODEL,
                "qualityThreshold": QUALITY_THRESHOLD,
                "maxReflectionAttempts": MAX_ATTEMPTS,
            },
            "pipelineConfiguration": {
                "actors": {
                    "promptingMode": ACTOR_PROMPTING_MODE.name,
                    "evaluator": False,
                },
                "highLevelGoals": {
                    "promptingMode": GOAL_PROMPTING_MODE.name,
                    "evaluator": True,
                },
                "lowLevelGoals": {
                    "promptingMode": GOAL_PROMPTING_MODE.name,
                    "evaluator": True,
                },
                "profile": BASE_CONFIG_NAME,
            },
        }

        output_file = top_down_output_file_for(dataset_name)
        with output_file.open("w", encoding="utf-8") as file:
            json.dump(
                output,
                file,
                indent=4,
                ensure_ascii=False,
                default=str,
            )

        print(
            f"[{dataset_name}] saved: {output_file.name} | "
            f"Actors={len(actors.actors)} (Few-shot, NO evaluator) | "
            f"HLG={len(high_level_goals.goals)} (evaluator ON) | "
            f"LLG={len(low_level_goals.low_level_goals)} (evaluator ON) | "
            f"evaluator={llm_clients.GROQ_EVALUATOR_MODEL}"
        )

    except Exception as error:
        baseline_errors[dataset_name] = (
            f"{type(error).__name__}: {error}"
        )
        print(
            f"[{dataset_name}] Top-down extraction failed: "
            f"{type(error).__name__}: {error}"
        )

        import traceback
        traceback.print_exc()


for groundtruth in GROUNDTRUTHS:
    run_top_down_for_dataset(groundtruth)


[SIA Project 25 26] Documentation ready | raw=5910 chars | pipeline=5910 chars
Actors STARTING... (Few-shot, no evaluator)
No feedback provided!
Actors DONE...
actors=[Actor(name='Citizens', description='Individuals who report issues and malfunctions in the urban environment.'), Actor(name='Municipal Operators', description='Staff members of the Municipality of Turin who review and manage citizen reports.'), Actor(name='External Maintenance Personnel', description='Workers from external companies who address specific issues reported by citizens.'), Actor(name='Administrators', description='Individuals who manage the Participium system and oversee its operations.')]
High Level Goals STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks.You're tasked to extract high level goals from a software description for each provided actor that is expected to interact with the software.Following the Goal-Orie

## 3. Riepilogo

Al termine viene mostrato il numero di dataset completati e falliti. Ogni dataset completato produce un JSON separato in `output/top_down/`.


In [6]:
print("\nTop-down execution summary")
print("--------------------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Completed datasets: {len(generated_ll_objects)}")
print(f"Failed datasets: {len(baseline_errors)}")
print(f"Output directory: {TOP_DOWN_DIR}")

if baseline_errors:
    print("Errors:")
    for dataset_name, error in baseline_errors.items():
        print(f"- {dataset_name}: {error}")


Top-down execution summary
--------------------------
Configured datasets: 4
Completed datasets: 4
Failed datasets: 0
Output directory: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down
